In [ ]:
# split_components.py
# Run this from inside your Downloads/Components folder

import os
import shutil
import random

src = r"D:\Downloads\Components"  # change this path
dst = r"D:\Downloads\Components_split"  # output folder

splits = {"train": 0.70, "val": 0.15, "test": 0.15}

for class_name in os.listdir(src):
    class_path = os.path.join(src, class_name)
    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    random.shuffle(images)

    n = len(images)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)

    split_data = {
        "train": images[:n_train],
        "val":   images[n_train:n_train + n_val],
        "test":  images[n_train + n_val:]
    }

    for split, files in split_data.items():
        out_dir = os.path.join(dst, split, class_name)
        os.makedirs(out_dir, exist_ok=True)
        for f in files:
            shutil.copy2(os.path.join(class_path, f), os.path.join(out_dir, f))
        print(f"{split}/{class_name}: {len(files)} images")

print("\nComponents split done!")

train/armature: 224 images
val/armature: 48 images
test/armature: 49 images
train/attenuator: 189 images
val/attenuator: 40 images
test/attenuator: 42 images
train/Bypass-capacitor: 211 images
val/Bypass-capacitor: 45 images
test/Bypass-capacitor: 46 images
train/cartridge-fuse: 157 images
val/cartridge-fuse: 33 images
test/cartridge-fuse: 35 images
train/clip-lead: 201 images
val/clip-lead: 43 images
test/clip-lead: 44 images
train/electric-relay: 294 images
val/electric-relay: 63 images
test/electric-relay: 63 images
train/Electrolytic-capacitor: 282 images
val/Electrolytic-capacitor: 60 images
test/Electrolytic-capacitor: 61 images
train/filament: 280 images
val/filament: 60 images
test/filament: 60 images
train/heat-sink: 294 images
val/heat-sink: 63 images
test/heat-sink: 63 images
train/images: 0 images
val/images: 0 images
test/images: 0 images
train/induction-coil: 105 images
val/induction-coil: 22 images
test/induction-coil: 24 images
train/Integrated-micro-circuit: 329 images

In [5]:
# fix_wastes_val.py
# Run this FIRST, before merge_datasets.py

import os
import shutil
import random

wastes_train = r"D:\Downloads\Wastes\train"  # update path
wastes_val   = r"D:\Downloads\Wastes\val"    # will be created

random.seed(42)  # ensures reproducibility

for class_name in os.listdir(wastes_train):
    class_path = os.path.join(wastes_train, class_name)
    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

    random.shuffle(images)

    # Take 15% from train → move to val
    n_val = max(1, int(len(images) * 0.15))
    val_images = images[:n_val]

    val_class_dir = os.path.join(wastes_val, class_name)
    os.makedirs(val_class_dir, exist_ok=True)

    for img in val_images:
        src = os.path.join(class_path, img)
        dst = os.path.join(val_class_dir, img)
        shutil.move(src, dst)  # moves out of train into val

    print(f"{class_name}: moved {n_val} images to val, {len(images)-n_val} remain in train")

print("\nWastes val folder created successfully!")

battery waste: moved 127 images to val, 721 remain in train
e-waste: moved 187 images to val, 1061 remain in train
light bulbs: moved 63 images to val, 357 remain in train

Wastes val folder created successfully!


In [6]:
# verify_current.py
import os

master = r"D:\Downloads\MASTER_DATASET"  # update path

print(f"\n{'Class':<35} {'Train':>7} {'Val':>7} {'Test':>7}  Status")
print("-" * 75)

total_train = total_val = total_test = 0

train_path = os.path.join(master, "train")
if not os.path.exists(train_path):
    print("MASTER_DATASET train folder not found — check path")
    exit()

classes = sorted(os.listdir(train_path))

for cls in classes:
    counts = {}
    for split in ["train", "val", "test"]:
        p = os.path.join(master, split, cls)
        counts[split] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))
        ]) if os.path.exists(p) else 0

    total_train += counts["train"]
    total_val   += counts["val"]
    total_test  += counts["test"]

    if counts["train"] == 0:
        status = "❌ EMPTY — delete"
    elif counts["train"] < 100:
        status = "🔴 CRITICAL — drop"
    elif counts["train"] < 200:
        status = "🟡 LOW — augment"
    elif counts["train"] < 300:
        status = "🟠 ACCEPTABLE"
    else:
        status = "🟢 GOOD"

    print(f"{cls:<35} {counts['train']:>7} {counts['val']:>7} "
          f"{counts['test']:>7}  {status}")

print("-" * 75)
print(f"{'TOTAL':<35} {total_train:>7} {total_val:>7} {total_test:>7}")
print(f"\nTotal classes : {len(classes)}")
print(f"Total images  : {total_train + total_val + total_test}")


Class                                 Train     Val    Test  Status
---------------------------------------------------------------------------
Battery                                 961     157     243  🟢 GOOD
Bypass-capacitor                        211      45      46  🟠 ACCEPTABLE
Electrolytic-capacitor                  282      60      61  🟠 ACCEPTABLE
Integrated-micro-circuit                329      70      71  🟢 GOOD
Keyboard                                240      30      30  🟠 ACCEPTABLE
LED                                     336      72      72  🟢 GOOD
Microwave                               240      30      30  🟠 ACCEPTABLE
Mobile                                  240      30      30  🟠 ACCEPTABLE
Mouse                                   240      30      30  🟠 ACCEPTABLE
PCB                                     240      30      30  🟠 ACCEPTABLE
PNP-transistor                          214      46      47  🟠 ACCEPTABLE
Player                                  240      30      30

In [7]:
# step1_drop_classes.py
import os, shutil

master = r"D:\Downloads\MASTER_DATASET"

DROP = [
    "light-circuit",
    "local-oscillator", 
    "multiplexer",
    "omni-directional-antenna",
    "shunt",
    "images"
]

for split in ["train", "val", "test"]:
    for cls in DROP:
        path = os.path.join(master, split, cls)
        if os.path.exists(path):
            shutil.rmtree(path)
            print(f"Deleted: {split}/{cls}")

print("\nDone — critical classes removed!")

Deleted: train/light-circuit
Deleted: train/local-oscillator
Deleted: train/multiplexer
Deleted: train/omni-directional-antenna
Deleted: train/shunt
Deleted: train/images
Deleted: val/light-circuit
Deleted: val/local-oscillator
Deleted: val/multiplexer
Deleted: val/omni-directional-antenna
Deleted: val/shunt
Deleted: val/images
Deleted: test/light-circuit
Deleted: test/local-oscillator
Deleted: test/multiplexer
Deleted: test/omni-directional-antenna
Deleted: test/shunt
Deleted: test/images

Done — critical classes removed!


In [5]:
# step2_augment.py — FIXED for albumentations >= 1.4
import os, cv2, random
import albumentations as A

master = r"D:\Downloads\MASTER_DATASET"

TARGETS = {
    # 🟡 LOW — boost to 300
    "attenuator":          300,
    "cartridge-fuse":      300,
    "induction-coil":      300,
    "jumper-cable":        300,
    "potential-divider":   300,
    "rheostat":            300,
    "stabilizer":          300,
    "step-up-transformer": 300,
    "transistor":          300,
    # 🟠 ACCEPTABLE — mild boost to 280
    "Bypass-capacitor":    280,
    "PNP-transistor":      280,
    "armature":            280,
    "clip-lead":           280,
    "memory-chip":         280,
}

strong_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.3,
        contrast_limit=0.3, p=0.5),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(p=0.2),
    A.RandomResizedCrop(
        size=(224, 224),        # ← FIXED
        scale=(0.75, 1.0), p=0.5),
])

mild_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2, p=0.4),
    A.RandomResizedCrop(
        size=(224, 224),        # ← FIXED
        scale=(0.85, 1.0), p=0.3),
])

for cls, target in TARGETS.items():
    train_path = os.path.join(master, "train", cls)
    if not os.path.exists(train_path):
        print(f"Skipping {cls} — folder not found")
        continue

    images = [f for f in os.listdir(train_path)
              if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]

    current = len(images)
    needed  = max(0, target - current)

    if needed == 0:
        print(f"{cls}: already at {current}, skipping")
        continue

    is_low    = current < 200
    transform = strong_transform if is_low else mild_transform

    print(f"{cls}: {current} → generating {needed} images...")

    aug_count = 0
    while aug_count < needed:
        src_file = random.choice(images)
        src_path = os.path.join(train_path, src_file)
        img = cv2.imread(src_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = transform(image=img)["image"]
        augmented = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        out_name  = f"aug_{aug_count:04d}_{src_file}"
        cv2.imwrite(os.path.join(train_path, out_name), augmented)
        aug_count += 1

    print(f"  Done — {cls} now has {current + aug_count} train images")

print("\nAugmentation complete!")

attenuator: 189 → generating 111 images...
  Done — attenuator now has 300 train images
cartridge-fuse: 157 → generating 143 images...
  Done — cartridge-fuse now has 300 train images
induction-coil: 105 → generating 195 images...
  Done — induction-coil now has 300 train images
jumper-cable: 174 → generating 126 images...
  Done — jumper-cable now has 300 train images
potential-divider: 165 → generating 135 images...
  Done — potential-divider now has 300 train images
rheostat: 118 → generating 182 images...
  Done — rheostat now has 300 train images
stabilizer: 113 → generating 187 images...
  Done — stabilizer now has 300 train images
step-up-transformer: 108 → generating 192 images...
  Done — step-up-transformer now has 300 train images
transistor: 107 → generating 193 images...
  Done — transistor now has 300 train images
Bypass-capacitor: 211 → generating 69 images...
  Done — Bypass-capacitor now has 280 train images
PNP-transistor: 214 → generating 66 images...
  Done — PNP-tr

In [10]:
# step3_final_verify.py
import os

master = r"D:\Downloads\MASTER_DATASET"

print(f"\n{'Class':<35} {'Train':>7} {'Val':>7} {'Test':>7}  Status")
print("-" * 75)

total_train = total_val = total_test = 0
classes = sorted(os.listdir(os.path.join(master, "train")))

for cls in classes:
    counts = {}
    for split in ["train", "val", "test"]:
        p = os.path.join(master, split, cls)
        counts[split] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))
        ]) if os.path.exists(p) else 0

    total_train += counts["train"]
    total_val   += counts["val"]
    total_test  += counts["test"]

    if counts["train"] == 0:
        status = "❌ EMPTY"
    elif counts["train"] < 200:
        status = "🟡 LOW"
    elif counts["train"] < 300:
        status = "🟠 ACCEPTABLE"
    else:
        status = "🟢 GOOD"

    print(f"{cls:<35} {counts['train']:>7} {counts['val']:>7} "
          f"{counts['test']:>7}  {status}")

print("-" * 75)
print(f"{'TOTAL':<35} {total_train:>7} {total_val:>7} {total_test:>7}")
print(f"\nFinal classes : {len(classes)}")
print(f"Final images  : {total_train + total_val + total_test}")


Class                                 Train     Val    Test  Status
---------------------------------------------------------------------------
Battery                                 961     237     163  🟢 GOOD
Capacitor                               444      95      96  🟢 GOOD
Integrated-micro-circuit               1381     295     297  🟢 GOOD
Keyboard                                240      30      30  🟠 ACCEPTABLE
LED                                     406      87      87  🟢 GOOD
Microwave                               240      30      30  🟠 ACCEPTABLE
Mobile                                  240      30      30  🟠 ACCEPTABLE
Mouse                                   240      30      30  🟠 ACCEPTABLE
PCB                                     240      30      30  🟠 ACCEPTABLE
PNP-transistor                          280      46      47  🟠 ACCEPTABLE
Player                                  240      30      30  🟠 ACCEPTABLE
Printer                                 240      30      30  🟠 AC

In [8]:
# fix_val_gap.py
import os, shutil, random

master = r"D:\Downloads\MASTER_DATASET"

# Move excess test images → val to balance them
FIX_CLASSES = {
    "Battery":     80,   # move 80 from test → val
    "light bulbs": 30,   # move 30 from test → val
}

random.seed(42)

for cls, move_count in FIX_CLASSES.items():
    test_path = os.path.join(master, "test", cls)
    val_path  = os.path.join(master, "val",  cls)
    os.makedirs(val_path, exist_ok=True)

    test_images = [f for f in os.listdir(test_path)
                   if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]

    random.shuffle(test_images)
    to_move = test_images[:move_count]

    for img in to_move:
        shutil.move(
            os.path.join(test_path, img),
            os.path.join(val_path,  img)
        )

    print(f"{cls}: moved {move_count} images from test → val")

print("\nDone!")

Battery: moved 80 images from test → val
light bulbs: moved 30 images from test → val

Done!


In [9]:
# add_new_datasets.py
import os, shutil, random, cv2

random.seed(42)

MASTER      = r"D:\Downloads\MASTER_DATASET"
DATASET1    = r"D:\Downloads\dataset"
DATASET2    = r"D:\Downloads\dataset2"

# ── Step 1: Drop old capacitor classes from MASTER ──────────────
DROP_FROM_MASTER = ["Bypass-capacitor", "Electrolytic-capacitor"]

print("── Step 1: Removing old capacitor classes ──")
for cls in DROP_FROM_MASTER:
    for split in ["train", "val", "test"]:
        path = os.path.join(MASTER, split, cls)
        if os.path.exists(path):
            shutil.rmtree(path)
            print(f"  Deleted: {split}/{cls}")

# ── Step 2: Define class mapping ─────────────────────────────────
# Format: { "source_folder_name": "target_class_name_in_MASTER" }

DATASET1_MAP = {
    "Transistors":  "transistor",
    "Resistors":    "Resistor",
    "LEDs":         "LED",
    "Chips":        "microchip",
    "Capacitors":   "Capacitor",
}

DATASET2_MAP = {
    "Transformer":  "step-down-transformer",
    "Resistor":     "Resistor",
    "Inductor":     "induction-coil",
    "IC":           "Integrated-micro-circuit",
    "Diode":        "semiconductor-diode",
    "Capacitor":    "Capacitor",
}

SPLITS = {"train": 0.70, "val": 0.15, "test": 0.15}

def split_and_merge(src_root, class_map, dataset_name):
    print(f"\n── Processing {dataset_name} ──")
    for src_folder, target_class in class_map.items():
        src_path = os.path.join(src_root, src_folder)
        if not os.path.exists(src_path):
            print(f"  Skipping {src_folder} — folder not found")
            continue

        images = [f for f in os.listdir(src_path)
                  if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * 0.70)
        n_val   = int(n * 0.15)

        split_data = {
            "train": images[:n_train],
            "val":   images[n_train:n_train + n_val],
            "test":  images[n_train + n_val:]
        }

        for split, files in split_data.items():
            dst_dir = os.path.join(MASTER, split, target_class)
            os.makedirs(dst_dir, exist_ok=True)
            for f in files:
                src_img = os.path.join(src_path, f)
                dst_img = os.path.join(dst_dir, f"{dataset_name}_{f}")
                shutil.copy2(src_img, dst_img)

        print(f"  {src_folder} → {target_class}: "
              f"{split_data['train'].__len__()} train | "
              f"{split_data['val'].__len__()} val | "
              f"{split_data['test'].__len__()} test")

# ── Step 3: Run merge for both datasets ──────────────────────────
split_and_merge(DATASET1, DATASET1_MAP, "ds1")
split_and_merge(DATASET2, DATASET2_MAP, "ds2")

# ── Step 4: Final count report ───────────────────────────────────
print(f"\n── Final Dataset Summary ──")
print(f"\n{'Class':<35} {'Train':>7} {'Val':>7} {'Test':>7}  Status")
print("-" * 75)

total_train = total_val = total_test = 0
classes = sorted(os.listdir(os.path.join(MASTER, "train")))

for cls in classes:
    counts = {}
    for split in ["train", "val", "test"]:
        p = os.path.join(MASTER, split, cls)
        counts[split] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))
        ]) if os.path.exists(p) else 0

    total_train += counts["train"]
    total_val   += counts["val"]
    total_test  += counts["test"]

    if counts["train"] == 0:
        status = "❌ EMPTY"
    elif counts["train"] < 200:
        status = "🟡 LOW"
    elif counts["train"] < 300:
        status = "🟠 ACCEPTABLE"
    else:
        status = "🟢 GOOD"

    print(f"{cls:<35} {counts['train']:>7} {counts['val']:>7} "
          f"{counts['test']:>7}  {status}")

print("-" * 75)
print(f"{'TOTAL':<35} {total_train:>7} {total_val:>7} {total_test:>7}")
print(f"\nFinal classes : {len(classes)}")
print(f"Final images  : {total_train + total_val + total_test}")


── Step 1: Removing old capacitor classes ──
  Deleted: train/Bypass-capacitor
  Deleted: val/Bypass-capacitor
  Deleted: test/Bypass-capacitor
  Deleted: train/Electrolytic-capacitor
  Deleted: val/Electrolytic-capacitor
  Deleted: test/Electrolytic-capacitor

── Processing ds1 ──
  Transistors → transistor: 70 train | 15 val | 15 test
  Resistors → Resistor: 67 train | 14 val | 16 test
  LEDs → LED: 70 train | 15 val | 15 test
  Chips → microchip: 70 train | 15 val | 15 test
  Capacitors → Capacitor: 70 train | 15 val | 15 test

── Processing ds2 ──
  Transformer → step-down-transformer: 522 train | 112 val | 113 test
  Resistor → Resistor: 329 train | 70 val | 71 test
  Inductor → induction-coil: 185 train | 39 val | 41 test
  IC → Integrated-micro-circuit: 1052 train | 225 val | 226 test
  Diode → semiconductor-diode: 362 train | 77 val | 79 test
  Capacitor → Capacitor: 374 train | 80 val | 81 test

── Final Dataset Summary ──

Class                                 Train     Val  

In [11]:
# boost_acceptable_classes.py
import os, cv2, random
import albumentations as A

MASTER = r"D:\Downloads\MASTER_DATASET"
TARGET = 400

random.seed(42)

BOOST_CLASSES = [
    # 240 → 400
    "Keyboard", "Microwave", "Mobile", "Mouse", "PCB",
    "Player", "Printer", "Television", "Washing Machine",
    # 280–298 → 400
    "PNP-transistor", "armature", "clip-lead", "electric-relay",
    "filament", "heat-sink", "limiter-clipper", "memory-chip",
    "semi-conductor",
    # 221 → 400
    "pulse-generator", "solenoid",
]

strong_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(
        brightness_limit=0.3,
        contrast_limit=0.3, p=0.5),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(p=0.2),
    A.RandomResizedCrop(
        size=(224, 224),
        scale=(0.75, 1.0), p=0.5),
])

mild_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2, p=0.4),
    A.RandomResizedCrop(
        size=(224, 224),
        scale=(0.85, 1.0), p=0.3),
])

print(f"{'Class':<30} {'Before':>8} {'Added':>8} {'After':>8}")
print("-" * 60)

for cls in BOOST_CLASSES:
    train_path = os.path.join(MASTER, "train", cls)
    if not os.path.exists(train_path):
        print(f"{cls:<30} NOT FOUND — skipping")
        continue

    images = [f for f in os.listdir(train_path)
              if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))]

    current = len(images)
    needed  = max(0, TARGET - current)

    if needed == 0:
        print(f"{cls:<30} {current:>8} {'—':>8} {current:>8}  already OK")
        continue

    # Strong transform for 240 classes, mild for 280+ classes
    transform = strong_transform if current <= 250 else mild_transform

    aug_count = 0
    while aug_count < needed:
        src_file = random.choice(images)
        src_path = os.path.join(train_path, src_file)
        img = cv2.imread(src_path)
        if img is None:
            continue
        img       = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = transform(image=img)["image"]
        augmented = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        out_name  = f"aug_{aug_count:04d}_{src_file}"
        cv2.imwrite(os.path.join(train_path, out_name), augmented)
        aug_count += 1

    print(f"{cls:<30} {current:>8} {aug_count:>8} {current+aug_count:>8}")

print("-" * 60)
print("\nAugmentation complete — run verify script to confirm.")

Class                            Before    Added    After
------------------------------------------------------------
Keyboard                            240      160      400
Microwave                           240      160      400
Mobile                              240      160      400
Mouse                               240      160      400
PCB                                 240      160      400
Player                              240      160      400
Printer                             240      160      400
Television                          240      160      400
Washing Machine                     240      160      400
PNP-transistor                      280      120      400
armature                            280      120      400
clip-lead                           280      120      400
electric-relay                      294      106      400
filament                            280      120      400
heat-sink                           294      106      400
limiter-cli

In [2]:
# final_verify.py
import os

MASTER = r"D:\Downloads\MASTER_DATASET"

print(f"\n{'Class':<35} {'Train':>7} {'Val':>7} {'Test':>7}  Status")
print("-" * 75)

total_train = total_val = total_test = 0
classes = sorted(os.listdir(os.path.join(MASTER, "train")))

for cls in classes:
    counts = {}
    for split in ["train", "val", "test"]:
        p = os.path.join(MASTER, split, cls)
        counts[split] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))
        ]) if os.path.exists(p) else 0

    total_train += counts["train"]
    total_val   += counts["val"]
    total_test  += counts["test"]

    if counts["train"] < 300:
        status = "🟡 LOW"
    elif counts["train"] < 400:
        status = "🟠 ACCEPTABLE"
    else:
        status = "🟢 GOOD"

    print(f"{cls:<35} {counts['train']:>7} {counts['val']:>7} "
          f"{counts['test']:>7}  {status}")

print("-" * 75)
print(f"{'TOTAL':<35} {total_train:>7} {total_val:>7} {total_test:>7}")
print(f"\nFinal classes : {len(classes)}")
print(f"Final images  : {total_train + total_val + total_test}")


Class                                 Train     Val    Test  Status
---------------------------------------------------------------------------
Battery                                 961     237     163  🟢 GOOD
Capacitor                               444      95      96  🟢 GOOD
Integrated-micro-circuit               1381     295     297  🟢 GOOD
Keyboard                                400      30      30  🟢 GOOD
LED                                     406      87      87  🟢 GOOD
Microwave                               400      30      30  🟢 GOOD
Mobile                                  400      30      30  🟢 GOOD
Mouse                                   400      30      30  🟢 GOOD
PCB                                     400      30      30  🟢 GOOD
Player                                  400      30      30  🟢 GOOD
Printer                                 400      30      30  🟢 GOOD
Resistor                                396      84      87  🟠 ACCEPTABLE
Television                       

In [1]:
# path_a_cleanup.py
import os, shutil

MASTER = r"D:\Downloads\MASTER_DATASET"

KEEP = {
    "Battery", "PCB", "Mobile", "Television", "Microwave",
    "Washing Machine", "Printer", "Keyboard", "Mouse", "Player",
    "light bulbs", "Capacitor", "LED", "Resistor",
    "Integrated-micro-circuit", "semiconductor-diode", "transistor",
    "microchip", "microprocessor", "heat-sink"
}

print("── Dropping non-e-waste classes ──")
dropped = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(MASTER, split)
    if not os.path.exists(split_path):
        continue
    for cls in os.listdir(split_path):
        if cls not in KEEP:
            path = os.path.join(split_path, cls)
            if os.path.isdir(path):
                shutil.rmtree(path)
                if cls not in dropped:
                    dropped.append(cls)
                    print(f"  Dropped: {cls}")

print(f"\nDropped {len(dropped)} classes")
print(f"Remaining: {len(KEEP)} classes")

# ── Final count report ──────────────────────────────────────────
print(f"\n{'Class':<35} {'Train':>7} {'Val':>7} {'Test':>7}  Hazard")
print("-" * 80)

HAZARD = {
    "Battery":                  "🔴 HIGH",
    "PCB":                      "🔴 HIGH",
    "Mobile":                   "🔴 HIGH",
    "Television":               "🔴 HIGH",
    "light bulbs":              "🔴 HIGH",
    "Microwave":                "🟡 MEDIUM",
    "Washing Machine":          "🟡 MEDIUM",
    "Printer":                  "🟡 MEDIUM",
    "Player":                   "🟡 MEDIUM",
    "Capacitor":                "🟡 MEDIUM",
    "Integrated-micro-circuit": "🟡 MEDIUM",
    "microchip":                "🟡 MEDIUM",
    "microprocessor":           "🟡 MEDIUM",
    "Keyboard":                 "🟢 LOW",
    "Mouse":                    "🟢 LOW",
    "LED":                      "🟢 LOW",
    "Resistor":                 "🟢 LOW",
    "semiconductor-diode":      "🟢 LOW",
    "transistor":               "🟢 LOW",
    "heat-sink":                "🟢 LOW",
}

total_train = total_val = total_test = 0
classes = sorted(os.listdir(os.path.join(MASTER, "train")))

for cls in classes:
    counts = {}
    for split in ["train", "val", "test"]:
        p = os.path.join(MASTER, split, cls)
        counts[split] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))
        ]) if os.path.exists(p) else 0

    total_train += counts["train"]
    total_val   += counts["val"]
    total_test  += counts["test"]

    hazard = HAZARD.get(cls, "❓ Unknown")
    print(f"{cls:<35} {counts['train']:>7} {counts['val']:>7} "
          f"{counts['test']:>7}  {hazard}")

print("-" * 80)
print(f"{'TOTAL':<35} {total_train:>7} {total_val:>7} {total_test:>7}")
print(f"\nFinal classes : {len(classes)}")
print(f"Final images  : {total_train + total_val + total_test}")

# ── Hazard distribution summary ─────────────────────────────────
print(f"\n── Hazard Distribution ──")
high   = [c for c in HAZARD if HAZARD[c] == "🔴 HIGH"]
medium = [c for c in HAZARD if HAZARD[c] == "🟡 MEDIUM"]
low    = [c for c in HAZARD if HAZARD[c] == "🟢 LOW"]
print(f"  🔴 HIGH   : {len(high)}  classes — {', '.join(high)}")
print(f"  🟡 MEDIUM : {len(medium)} classes — {', '.join(medium)}")
print(f"  🟢 LOW    : {len(low)}  classes — {', '.join(low)}")

── Dropping non-e-waste classes ──
  Dropped: armature
  Dropped: attenuator
  Dropped: cartridge-fuse
  Dropped: clip-lead
  Dropped: electric-relay
  Dropped: filament
  Dropped: induction-coil
  Dropped: jumper-cable
  Dropped: junction-transistor
  Dropped: limiter-clipper
  Dropped: memory-chip
  Dropped: PNP-transistor
  Dropped: potential-divider
  Dropped: potentiometer
  Dropped: pulse-generator
  Dropped: relay
  Dropped: rheostat
  Dropped: semi-conductor
  Dropped: solenoid
  Dropped: stabilizer
  Dropped: step-down-transformer
  Dropped: step-up-transformer

Dropped 22 classes
Remaining: 20 classes

Class                                 Train     Val    Test  Hazard
--------------------------------------------------------------------------------
Battery                                 961     237     163  🔴 HIGH
Capacitor                               444      95      96  🟡 MEDIUM
Integrated-micro-circuit               1381     295     297  🟡 MEDIUM
Keyboard               